# Assignment — Bloc 2
### Mini-repte: "Audio Manipulator" (loop + efectes bàsics)

**Com obrir aquest notebook:** Fes clic a l'enllaç del Classroom. Colab el carregarà directament des de GitHub. Per guardar els teus canvis: `File -> Save a copy in Drive`.

**Objectiu:** Construeix un petit "manipulador" que llegeix un so, l'allarga amb un loop, hi aplica gain i fade, i el barreja amb un segon so. Completa les funcions marcades amb `# TODO`. Cada part té una cel·la d'autotest.

No esborris les cel·les d'autotest.

In [ ]:
import numpy as np
import soundfile as sf
import librosa
import librosa.display
import matplotlib.pyplot as plt
from IPython.display import Audio

sample_rate = 44100

## Part 0 — El teu so d'entrada

**Opció A (recomanada): grava la teva veu.** Executa la cel·la següent, dona permís al micròfon, i grava 2-3 segons (un so curt, una paraula, un "beatbox", el que vulguis).

**Opció B (si tens problemes amb el micròfon):** salta la cel·la de gravació i fes servir un dels fitxers d'exemple proporcionats (cel·la "Opció B").

### Opció A — Gravar amb el micròfon

In [ ]:
# Si dona problemes, salta aquesta cel·la i usa l'Opció B més avall

from google.colab import output
from base64 import b64decode
import io

RECORD = """
const sleep = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})
var recorder, stream
async function record(seconds) {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true })
  recorder = new MediaRecorder(stream)
  const chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  await sleep(seconds * 1000)
  recorder.stop()
  await new Promise(resolve => recorder.onstop = resolve)
  stream.getTracks().forEach(t => t.stop())
  const blob = new Blob(chunks)
  return await b2text(blob)
}
"""

def record_audio(seconds=3):
    from IPython.display import Javascript, display
    display(Javascript(RECORD))
    from google.colab import output as colab_output
    s = colab_output.eval_js(f'record({seconds})')
    b = b64decode(s.split(',')[1])
    with open('input_raw.webm', 'wb') as f:
        f.write(b)
    return 'input_raw.webm'

print("Gravant 3 segons... parla/canta/fes un so ara!")
record_audio(seconds=3)

# Convertim de webm a wav
!ffmpeg -y -i input_raw.webm -ar 44100 -ac 1 input.wav -loglevel quiet
print("Gravat com input.wav")

### Opció B — Usar un fitxer d'exemple proporcionat

In [ ]:
# Executa NOMÉS si l'Opció A no t'ha funcionat

import urllib.request

base_url = "https://raw.githubusercontent.com/jcomajuncosas/tp2/main/02_numpy_audio/sessio02_wav_io/recursos/audio/"
urllib.request.urlretrieve(base_url + "example_pad.wav", "input.wav")
print("Fitxer d'exemple descarregat com input.wav")

In [ ]:
# Comprovem que tenim un input.wav, sigui de l'Opció A o B

input_data, input_sr = librosa.load("input.wav", sr=sample_rate)

print("Durada:", len(input_data) / sample_rate, "segons")
librosa.display.waveshow(input_data, sr=sample_rate)
plt.title("El teu so d'entrada")
plt.show()

Audio(input_data, rate=sample_rate)

## Part 1 — Gain

Completa `apply_gain`: ha de tornar l'array `data` multiplicat per `factor`.

In [ ]:
def apply_gain(data, factor):
    # TODO 1: retorna 'data' multiplicat per 'factor'
    return None  # <-- substitueix

In [ ]:
# AUTOTEST 1 — no modifiquis aquesta cel·la

test_data = np.array([0.4, -0.4, 0.2])
result = apply_gain(test_data, 0.5)

assert result is not None, "La funció retorna None: revisa el TODO"
assert np.allclose(result, np.array([0.2, -0.2, 0.1])), f"Resultat incorrecte: {result}"

print("✅ Part 1 correcta!")

quiet = apply_gain(input_data, 0.3)
Audio(quiet, rate=sample_rate)

## Part 2 — Fade out

Completa `apply_fade_out`: l'última part del so (`fade_duration` segons) ha de baixar gradualment a 0.

**Pista:** crea un array `envelope` ple d'uns (`np.ones`), i substitueix els últims `fade_samples` valors per un `np.linspace(1, 0, fade_samples)`. Després multiplica `data * envelope`.

In [ ]:
def apply_fade_out(data, fade_duration, sample_rate=44100):
    fade_samples = int(fade_duration * sample_rate)
    fade_samples = min(fade_samples, len(data))

    envelope = np.ones(len(data))

    # TODO 2: substitueix els ULTIMS 'fade_samples' valors de 'envelope'
    # per un np.linspace(1, 0, fade_samples)
    # Pista: envelope[-fade_samples:] = ...

    return data * envelope

In [ ]:
# AUTOTEST 2 — no modifiquis aquesta cel·la

test_data = np.ones(1000)
result = apply_fade_out(test_data, fade_duration=0.01, sample_rate=44100)  # 441 mostres

assert len(result) == 1000, f"Longitud incorrecta: {len(result)}"
assert np.isclose(result[0], 1.0), f"L'inici hauria de seguir a 1.0, és {result[0]}"
assert result[-1] < 0.01, f"El final hauria de ser ~0, és {result[-1]}"
assert result[500] > result[-1], "L'envolvent hauria de decreixer cap al final"

print("✅ Part 2 correcta!")

faded = apply_fade_out(input_data, fade_duration=0.5, sample_rate=sample_rate)
Audio(faded, rate=sample_rate)

## Part 3 — Loop

Completa `make_loop`: ha de repetir `data` `n_repeats` vegades.

In [ ]:
def make_loop(data, n_repeats):
    # TODO 3: retorna 'data' repetit 'n_repeats' vegades (np.tile)
    return None  # <-- substitueix

In [ ]:
# AUTOTEST 3 — no modifiquis aquesta cel·la

test_data = np.array([1, 2, 3])
result = make_loop(test_data, 3)

assert result is not None, "La funció retorna None: revisa el TODO"
assert len(result) == 9, f"S'esperaven 9 elements, n'hi ha {len(result)}"
assert np.array_equal(result, np.array([1,2,3,1,2,3,1,2,3])), f"Resultat incorrecte: {result}"

print("✅ Part 3 correcta!")

looped = make_loop(input_data, 3)
print("Durada del loop:", len(looped)/sample_rate, "segons")
Audio(looped, rate=sample_rate)

## Part 4 — Mixing amb el loop de percussió

Completa `mix_sounds`: rep dos arrays de longituds potencialment diferents, els retalla a la mateixa longitud (la del més curt), els suma i normalitza el resultat perquè l'amplitud màxima sigui 1.

In [ ]:
# Descarreguem el loop de percussio per usar-lo en el mixing
import urllib.request

base_url = "https://raw.githubusercontent.com/jcomajuncosas/tp2/main/02_numpy_audio/sessio02_wav_io/recursos/audio/"
urllib.request.urlretrieve(base_url + "perc_loop.wav", "perc_loop.wav")

perc, sr_perc = librosa.load("perc_loop.wav", sr=sample_rate)
print("Loop de percussio carregat, durada:", len(perc)/sample_rate, "s")

In [ ]:
def mix_sounds(data1, data2):
    # TODO 4a: calcula 'n' com el minim entre len(data1) i len(data2)
    n = None  # <-- substitueix

    # TODO 4b: suma els dos arrays retallats a longitud 'n'
    mix = None  # <-- substitueix

    # TODO 4c: normalitza 'mix' dividint pel seu valor absolut maxim
    mix = None  # <-- substitueix

    return mix

In [ ]:
# AUTOTEST 4 — no modifiquis aquesta cel·la

test_a = np.array([0.5, 0.5, 0.5, 0.5, 0.5])
test_b = np.array([0.5, 0.5, 0.5])
result = mix_sounds(test_a, test_b)

assert result is not None, "La funció retorna None: revisa els TODO"
assert len(result) == 3, f"S'esperava longitud 3 (la del més curt), és {len(result)}"
assert np.isclose(np.max(np.abs(result)), 1.0), f"El maxim hauria de ser 1.0, es {np.max(np.abs(result))}"

print("✅ Part 4 correcta!")

# Apliquem tot el que hem fet: loop del nostre so + percussio
my_loop = make_loop(input_data, 2)
perc_loop = make_loop(perc, 2)

final_mix = mix_sounds(apply_gain(my_loop, 1.0), apply_gain(perc_loop, 0.7))
final_mix = apply_fade_out(final_mix, fade_duration=1.0, sample_rate=sample_rate)

librosa.display.waveshow(final_mix, sr=sample_rate)
plt.title("Resultat final: el teu so + percussio, amb fade")
plt.show()

Audio(final_mix, rate=sample_rate)

---
## 🚀 Challenges (opcional, nivell avançat)

1. **Reverse**: afegeix una funció `reverse(data)` que retorni el so al revés (`data[::-1]`). Combina-ho amb el mix final.
2. **Crossfade**: en lloc de sumar directament `my_loop` i `perc_loop`, fes que `perc_loop` entri gradualment (fade in) mentre `my_loop` ja sona.
3. **Pitch shift simple**: prova de canviar la velocitat de reproducció (`Audio(data, rate=sample_rate*1.5)`) — què passa amb el to i la durada? Per què creus que passa (pensa en la Sessió 1: què representa `sample_rate`)?
4. **Desa el resultat**: usa `sf.write` per desar `final_mix` com a `.wav` i descarrega't el fitxer (`from google.colab import files; files.download("resultat.wav")`).

In [ ]:
# El teu codi del Challenge aquí